In [6]:
import sys
from pathlib import Path
import requests
import string
import secrets
import time
import re
import collections
import numpy as np
import pandas as pd
import pickle

try:
    from urllib.parse import parse_qs, urlencode, urlparse
except ImportError:
    from urlparse import parse_qs, urlparse
    from urllib import urlencode

from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)


PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

    


In [7]:
from hangman.models.presence_model import PresenceModel
from hangman.models.masked_letter_model import MaskedLetterModel


In [8]:
DICT_PATH = PROJECT_ROOT / "data" / "words_250000_train.txt"

PRESENCE_MODEL_PATH = PROJECT_ROOT / "src" / "hangman" / "models" / "presence_model.pkl"
MASK_MODEL_PATH = PROJECT_ROOT /"src" / "hangman" / "models" / "mask_model.pkl"

presence_model = PresenceModel.load(
    model_path=PRESENCE_MODEL_PATH,
    dictionary_path=DICT_PATH
)

masked_model = MaskedLetterModel.load(
    model_path=MASK_MODEL_PATH,
    dictionary_path=DICT_PATH
)

print("Models loaded successfully")


Models loaded successfully


In [12]:
class HangmanAPI_ML(object):
    
    def __init__(self, presence_model, masked_model, access_token=None, session=None, timeout=None):
        self.hangman_url = self.determine_hangman_url()
        self.access_token = access_token
        self.session = session or requests.Session()
        self.timeout = timeout
        self.guessed_letters = []
        self.wrong_letters = set()

        self.presence_model = presence_model
        self.mask_model = masked_model

        self.fx = self.presence_model.fx



        self.stats = {
        'total_guesses': 0,
        'vowel_first': 0,
        'presence_ml': 0,
        'mask_ml': 0,
        'fallback': 0
    }

    
    @staticmethod
    def determine_hangman_url():
        links = ['https://trexsim.com']
        data = {link: 0 for link in links}
        for link in links:
            requests.get(link)
            for i in range(10):
                s = time.time()
                requests.get(link)
                data[link] = time.time() - s
        link = sorted(data.items(), key=lambda x: x[1])[0][0]
        link += '/trexsim/hangman'
        return link

    def _get_blank_positions(self, word_state):
        clean_word = word_state[::2]  # Remove spaces
        blank_positions = []
        for i, char in enumerate(clean_word):
            if char == '_':
                blank_positions.append(i)
        return blank_positions
    
    
    def _get_position_letter_frequencies(self, candidates, positions):
        letter_counter = collections.Counter()
        
        for word in candidates:
            for pos in positions:
                if pos < len(word):
                    letter = word[pos]
                    letter_counter[letter] += 1
        
        return letter_counter



    def _endgame_beam_guess(self, clean, blanks, unguessed, presence_scores=None):
        topk = 7
        L = len(clean)
        presence_scores = presence_scores or {}
    
        def bigram_prev_prob(prev_ch, cur_ch):
            # P(cur | prev) using bigram_prev / prev_total with add-one smoothing
            num = self.fx.bigram_prev.get((prev_ch, cur_ch), 0)
            den = self.fx.prev_total.get(prev_ch, 0)
            return (num + 1.0) / (den + 26.0)
    
        def bigram_next_prob(cur_ch, next_ch):
            # P(next | cur) using bigram_next / out_total with add-one smoothing
            num = self.fx.bigram_next.get((cur_ch, next_ch), 0)
            den = self.fx.out_total.get(cur_ch, 0)
            return (num + 1.0) / (den + 26.0)
    
        # Get topK options per blank from mask model
        per_pos = []
        for pos in blanks:
    
            pairs = []
            for c in unguessed:
                p = self.mask_model.score_letter(clean, pos, c)
                pairs.append((c, p))
            
            pairs = sorted(pairs, key=lambda x: x[1], reverse=True)[:topk]
            per_pos.append((pos, pairs))

        # 1 blank: choose best letter by mask * presence
        if len(blanks) == 1:
            pos, pairs = per_pos[0]
            left = "^" if pos == 0 else clean[pos - 1]
            right = "$" if pos == L - 1 else clean[pos + 1]
    
            best_ch, best_sc = None, -1.0
            for ch, p in pairs:
                if ch not in unguessed:
                    continue
    
                # Local n-gram plausibility (helps when mask is uncertain)
                ngram = 1.0
                if left != "_":
                    ngram *= bigram_prev_prob(left, ch)
                if right != "_":
                    ngram *= bigram_next_prob(ch, right)
    
                # Presence makes it less likely to pick a weird low-presence letter as the final guess
                pres = presence_scores.get(ch, 1.0)
                sc = float(p) * ngram * (pres ** 0.5)
    
                if sc > best_sc:
                    best_sc = sc
                    best_ch = ch
    
            return best_ch
    
        # 2 blanks: enumerate combinationsadd a bridge when blanks are adjacent 
        if len(blanks) == 2:
            (pos1, pairs1), (pos2, pairs2) = per_pos[0], per_pos[1]
            # ensure pos1 < pos2
            if pos2 < pos1:
                pos1, pos2 = pos2, pos1
                pairs1, pairs2 = pairs2, pairs1
    
            adjacent = (pos2 == pos1 + 1)
    
            left1 = "^" if pos1 == 0 else clean[pos1 - 1]
            right2 = "$" if pos2 == L - 1 else clean[pos2 + 1]
    
            best_completion = None
            best_sc = -1.0
    
            for c1, p1 in pairs1:
                for c2, p2 in pairs2:
                    # base (mask model)
                    sc = float(p1) * float(p2)
    
                    # n-gram bridge:only if adjacent and neighbors known
                    if adjacent:
                        ngram = 1.0
                        if left1 != "_":
                            ngram *= bigram_prev_prob(left1, c1)
                        ngram *= bigram_next_prob(c1, c2)
                        if right2 != "_":
                            ngram *= bigram_next_prob(c2, right2)
                        sc *= ngram
    
                    # slight preference for letters with higher presence probability
                    sc *= (presence_scores.get(c1, 1.0) ** 0.25) * (presence_scores.get(c2, 1.0) ** 0.25)
    
                    if sc > best_sc:
                        best_sc = sc
                        best_completion = (c1, c2)
    
            if not best_completion:
                return None
    
            c1, c2 = best_completion
    
            # Choose which letter to guess next (risk-aware)
            # Prefer the letter with higher presence, but only among unguessed.
            options = [ch for ch in (c1, c2) if ch in unguessed]
            if not options:
                return None
            options.sort(key=lambda ch: presence_scores.get(ch, 0.0), reverse=True)
            return options[0]
    
        return None
    


    def guess(self, word: str) -> str:
        self.stats['total_guesses'] += 1
    
        clean = word[::2]
        L = len(clean)
        blanks = [i for i, c in enumerate(clean) if c == "_"]
        num_blanks = len(blanks)
        num_revealed = L - num_blanks
        completion = num_revealed / max(1, L)
    
        guessed = set(self.guessed_letters)
        wrong = set(self.wrong_letters)
        unguessed = [c for c in string.ascii_lowercase if c not in guessed]

        if L >= 7:
            if len(self.guessed_letters) == 0 and 'e' in unguessed:
                self.stats['vowel_first'] += 1
                return 'e'
            if len(self.guessed_letters) == 1 and 'a' in unguessed:
                self.stats['vowel_first'] += 1
                return 'a'

    
        # 1) Presence model score for every unguessed letter 
        presence_scores = {}
        if self.presence_model is not None:
            for cand in unguessed:
                # feats = self.fx.extract_presence_features(
                #     word_state=word,
                #     guessed_letters=guessed,
                #     wrong_letters=wrong,
                #     candidate_letter=cand
                # )
                # # predict_proba: column 1 is P(label=1)
                # p = float(self.presence_model.predict_proba([feats])[0][1])
                # presence_scores[cand] = p
                p = self.presence_model.score_letter(
                word_state=word,
                guessed=guessed,
                wrong=wrong,
                letter=cand
                )
                presence_scores[cand] = p

            self.stats['presence_ml'] += 1
        else:
            # If model missing, fall back to a safe heuristic based on global priors only
            for cand in unguessed:
                presence_scores[cand] = self.fx.global_letter_freq.get(cand, 0) / max(1, self.fx.global_total)


        #Endgame beam: override when 1–2 blanks 
        if self.mask_model is not None and (0 < num_blanks <= 2):
            ch = self._endgame_beam_guess(clean, blanks, set(unguessed), presence_scores=presence_scores)
            if ch is not None:
                # self.stats['mask_ml'] += 1
                return ch

        
        # 2) Masked-letter boost (short words + endgame) 
        # Compute an additional per-letter score from blank positions
        mask_scores = {c: 0.0 for c in unguessed}
        use_mask = (self.mask_model is not None) and (L <= 6 or num_blanks <= 2 or completion >= 0.7)


        if use_mask and blanks:
            # P(letter appears in at least one blank)
            prob_not = {c: 1.0 for c in unguessed}
        
            for pos in blanks:
                for c in unguessed:
                    p = self.mask_model.score_letter(clean, pos, c)
                    p = max(0.0, min(1.0, float(p)))
                    prob_not[c] *= (1.0 - p)
        
            for c in mask_scores:
                mask_scores[c] = 1.0 - prob_not[c]
        
            self.stats['mask_ml'] += 1


    
        # 3) Combine scores (phase-dependent weights) 
        if use_mask:
            # more weight to mask model when endgame/short word
            if num_blanks <= 2 or L <= 5:
                w_mask = 0.70
            elif completion >= 0.7:
                w_mask = 0.55
            else:
                w_mask = 0.35
        else:
            w_mask = 0.0
    
        w_presence = 1.0 - w_mask


        tries = getattr(self, "tries_remains", 6)
        

        if tries <= 2:
            
            p_min = 0.18 if tries == 2 else 0.28
        else:
            p_min = -1.0  # no filtering
        
        best_letter = None
        best_score = -1.0
        
        best_safe_letter = None
        best_safe_score = -1.0
        
        for cand in unguessed:
            score = w_presence * presence_scores.get(cand, 0.0) + w_mask * mask_scores.get(cand, 0.0)
        
            # best overall 
            if score > best_score:
                best_score = score
                best_letter = cand
        
            
            if presence_scores.get(cand, 0.0) >= p_min:
                if score > best_safe_score:
                    best_safe_score = score
                    best_safe_letter = cand
        
        if best_safe_letter is not None:
            return best_safe_letter
        
        if best_letter is not None:
            return best_letter

        self.stats['fallback'] += 1
        return 'e'

    
    def build_dictionary(self, dictionary_file_location):
        text_file = open(dictionary_file_location, "r")
        full_dictionary = text_file.read().splitlines()
        text_file.close()
        return full_dictionary
   
    def start_game(self, practice=True, verbose=True):
        # RESET for new game
        self.guessed_letters = []
        self.wrong_letters = set() 
        # self.current_dictionary = self.full_dictionary
        
        response = self.request("/new_game", {"practice":practice})
        if response.get('status')=="approved":
            game_id = response.get('game_id')
            word = response.get('word')
            tries_remains = response.get('tries_remains')
            if verbose:
                print("Successfully start a new game! Game ID: {0}. # of tries remaining: {1}. Word: {2}.".format(game_id, tries_remains, word))
            
            prev_word = word  #  Track previous word state
            
            while tries_remains>0:
                # Get guess
                self.tries_remains = tries_remains
                guess_letter = self.guess(word)
                
                
                # Record guess
                self.guessed_letters.append(guess_letter)
                if verbose:
                    print("Guessing letter: {0}".format(guess_letter))
                
                try: 
                    res = self.request("/guess_letter", {"request":"guess_letter", "game_id":game_id, "letter":guess_letter})
                except HangmanAPIError:
                    print('HangmanAPIError exception caught on request.')
                    continue
                except Exception as e:
                    print('Other exception caught on request.')
                    raise e
                
                if verbose:
                    print("Sever response: {0}".format(res))
                
                status = res.get('status')
                tries_remains = res.get('tries_remains')
                new_word = res.get('word')
                
                #  CHECK IF GUESS WAS WRONG
                if new_word == prev_word:
                    # Word didn't change → wrong guess
                    self.wrong_letters.add(guess_letter)
                    if verbose:
                        print(f" Wrong guess: {guess_letter}")
                else:
                    if verbose:
                        print(f" Correct guess: {guess_letter}")
                
                prev_word = new_word  # Update for next iteration
                
                if status=="success":
                    if verbose:
                        print("Successfully finished game: {0}".format(game_id))
                    return True
                elif status=="failed":
                    reason = res.get('reason', '# of tries exceeded!')
                    if verbose:
                        print("Failed game: {0}. Because of: {1}".format(game_id, reason))
                    return False
                elif status=="ongoing":
                    word = new_word
            else:
                if verbose:
                    print("Failed to start a new game")
            
            return status=="success"
        
    def my_status(self):
        return self.request("/my_status", {})
    
    def request(self, path, args=None, post_args=None, method=None):
        if args is None:
            args = dict()
        if post_args is not None:
            method = "POST"
        if self.access_token:
            if post_args and "access_token" not in post_args:
                post_args["access_token"] = self.access_token
            elif "access_token" not in args:
                args["access_token"] = self.access_token
        time.sleep(0.1)
        num_retry, time_sleep = 50, 1
        for it in range(num_retry):
            try:
                response = self.session.request(
                    method or "GET", self.hangman_url + path,
                    timeout=self.timeout, params=args,
                    data=post_args, verify=False
                )
                break
            except requests.HTTPError as e:
                response = json.loads(e.read())
                raise HangmanAPIError(response)
            except requests.exceptions.SSLError as e:
                if it + 1 == num_retry:
                    raise
                time.sleep(time_sleep)
        headers = response.headers
        if 'json' in headers['content-type']:
            result = response.json()
        elif "access_token" in parse_qs(response.text):
            query_str = parse_qs(response.text)
            if "access_token" in query_str:
                result = {"access_token": query_str["access_token"][0]}
                if "expires" in query_str:
                    result["expires"] = query_str["expires"][0]
            else:
                raise HangmanAPIError(response.json())
        else:
            raise HangmanAPIError('Maintype was not text, or querystring')
        if result and isinstance(result, dict) and result.get("error"):
            raise HangmanAPIError(result)

        
        
        return result
    
    def print_stats(self):
        tg = max(1, self.stats['total_guesses'])
        print(f"Total guesses: {self.stats['total_guesses']}")
        # print(f"Vowel-first: {self.stats.get('vowel_first', 0)} ({100*self.stats.get('vowel_first', 0)/tg:.1f}%)")
        print(f"Presence ML: {self.stats.get('presence_ml', 0)} ({100*self.stats.get('presence_ml', 0)/tg:.1f}%)")
        print(f"Mask ML: {self.stats.get('mask_ml', 0)} ({100*self.stats.get('mask_ml', 0)/tg:.1f}%)")
        # print(f"End-game (position): {self.stats.get('endgame', 0)} ({100*self.stats.get('endgame', 0)/tg:.1f}%)")
        print(f"Frequency fallback: {self.stats['fallback']} ({100*self.stats['fallback']/tg:.1f}%)")



class HangmanAPIError(Exception):
    def __init__(self, result):
        self.result = result
        self.code = None
        try:
            self.type = result["error_code"]
        except (KeyError, TypeError):
            self.type = ""

        try:
            self.message = result["error_description"]
        except (KeyError, TypeError):
            try:
                self.message = result["error"]["message"]
                self.code = result["error"].get("code")
                if not self.type:
                    self.type = result["error"].get("type", "")
            except (KeyError, TypeError):
                try:
                    self.message = result["error_msg"]
                except (KeyError, TypeError):
                    self.message = result

        Exception.__init__(self, self.message)

        
print("HangmanAPI_ML created")


HangmanAPI_ML created


In [ ]:
# Test Phase 1.5 improvements


api_ml = HangmanAPI_ML(
    presence_model=presence_model,
    masked_model=masked_model,
    access_token="82a29630d4319d7d9c7d40b66d3f21",
    timeout=2000
)
num_test_games = 200
wins = 0

for i in range(num_test_games):
    result = api_ml.start_game(practice=1, verbose=False)
    if result:
        wins += 1
    if (i+1) % 10 == 0:
        print(f"Progress: {i+1}/{num_test_games} games, win rate: {100*wins/(i+1):.1f}%")

success_rate = wins / num_test_games
print(f"{wins}/{num_test_games} wins = {success_rate*100:.1f}%")

api_ml.print_stats()


## Playing recorded games:
Please finalize your code prior to running the cell below. Once this code executes once successfully your submission will be finalized. Our system will not allow you to rerun any additional games.

Please note that it is expected that after you successfully run this block of code that subsequent runs will result in the error message "Your account has been deactivated".

Once you've run this section of the code your submission is complete. Please send us your source code via email.

In [ ]:
for i in range(1000):
    print('Playing ', i, ' th game')
    # Uncomment the following line to execute your final runs. Do not do this until you are satisfied with your submission
    #api.start_game(practice=0,verbose=False)
    
    # DO NOT REMOVE as otherwise the server may lock you out for too high frequency of requests
    time.sleep(0.5)